In [413]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/titanic_propre.csv")

print(df.shape)
df.head()

(891, 10)


,survived,pclass,sex,age,sibsp,parch,fare,who,embark_town,niveau_anomalie
0,0,3,male,22.0,1,0,7.2500,man,Southampton,Aucune
1,1,1,female,38.0,1,0,71.2833,woman,Cherbourg,Aucune
2,1,3,female,26.0,0,0,7.9250,woman,Southampton,Aucune
3,1,1,female,35.0,1,0,53.1000,woman,Southampton,Aucune
4,0,3,male,35.0,0,0,8.0500,man,Southampton,Aucune


Les femmes ont-elles mieux survécu que les hommes dans chaque classe ?

In [414]:
print(f"Nombre de survivants par classe:\n")

for pclass, survie in (
    df.groupby('pclass')['survived']
    .mean()
    .mul(100)
    .round(2)
    .items()
):
    nb_passagers = len(df[df['pclass'] == pclass])

    print(f"Classe {pclass} : {nb_passagers} passagers ({survie:.2f}%)")


print("\n\n% de survie par classe et par genre :")

display(
    df.pivot_table(
        values='survived',
        index='pclass',
        columns='sex',
        aggfunc='mean'
    ).mul(100).round(2)
)


Nombre de survivants par classe:

Classe 1 : 216 passagers (62.96%)
Classe 2 : 184 passagers (47.28%)
Classe 3 : 491 passagers (24.24%)


% de survie par classe et par genre :


sex,female,male
pclass,,
1,96.81,36.89
2,92.11,15.74
3,50.00,13.54


Nous constatons que les femmes ont mieux survécu que les hommes dans toutes les catégories.

Les enfants ont-ils été prioritaires ? Quel taux de survie par tranche d'âge?

In [415]:
print("% de survie par type de passager :\n")

for who, survie in (
    df.groupby('who')['survived']
    .mean()
    .mul(100)
    .round(2)
    .items()
):
    nb_passagers = len(df[df['who'] == who])

    print(f" {who} : {nb_passagers} passagers ({survie:.2f}%)")


print("\n\n% de survie par classe et par type de passager :")
display(
    df.pivot_table(
        values='survived',
        index='pclass',
        columns='who',
        aggfunc='mean'
    ).mul(100).round(2)
)


print("\n\n% de survie par tranche d'âge :")
df["tranche_age"] = pd.cut(df["age"], bins=[0, 15, 25, 60, float('inf')], labels=["enfant", "jeune adulte", "adulte", "senior"],)

display(
    df.pivot_table(
        values="survived",
        index="tranche_age",
        aggfunc="mean"
    )
    .mul(100)
    .round(2)
)


% de survie par type de passager :

 child : 83 passagers (59.04%)
 man : 537 passagers (16.39%)
 woman : 271 passagers (75.65%)


% de survie par classe et par type de passager :


who,child,man,woman
pclass,,,
1,83.33,35.29,97.80
2,100.00,8.08,90.91
3,43.10,11.91,49.12




% de survie par tranche d'âge :


,survived
tranche_age,
enfant,59.04
jeune adulte,30.79
adulte,41.44
senior,22.73


On remarque que les femmes et les enfants semblaient être prioritaires par rapport aux hommes. En analysant les taux de survie par tranche d'âge, on constate que les enfants présentent le taux de survie le plus élevé. Ces résultats suggèrent qu'ils ont bénéficié d'une évacuation prioritaire.

Quel port a le prix moyen le plus élevé ? La répartition des classes l'explique-t-elle ?

In [416]:
print("\nPrix moyen des billets par port d'embarquement en € :\n")
display(
    df.groupby('embark_town')['fare']
    .mean()
    .round(2)
    .to_frame('prix_moyen')
)

print("\nRépartition des passagers par port d'embarquement par rapport à la classe en % :\n")
display(
    pd.crosstab(
    df['embark_town'],
    df['pclass'],
    normalize='index'
).round(2)
)

print("\nPrix moyen des billets et répartition des passagers par port d'embarquement et par classe :\n")
tableau = pd.concat(
    [
        df.groupby('embark_town')['fare']
        .mean()
        .round(2)
        .rename('prix_moyen'),

        pd.crosstab(
            df['embark_town'],
            df['pclass'],
            normalize='index'
        )
        .mul(100)
        .round(2)
        .rename(columns={
            1: '1ere_classe',
            2: '2eme_classe',
            3: '3eme_classe'
        })
    ],
    axis=1
)

tableau['prix_moyen'] = tableau['prix_moyen'].map(lambda x: f"{x:.2f} €")

for col in ['1ere_classe', '2eme_classe', '3eme_classe']:
    tableau[col] = tableau[col].map(lambda x: f"{x:.2f} %")

display(tableau)


Prix moyen des billets par port d'embarquement en € :



,prix_moyen
embark_town,
Cherbourg,59.95
Inconnu,80.00
Queenstown,13.28
Southampton,27.08



Répartition des passagers par port d'embarquement par rapport à la classe en % :



pclass,1,2,3
embark_town,,,
Cherbourg,0.51,0.10,0.39
Inconnu,1.00,0.00,0.00
Queenstown,0.03,0.04,0.94
Southampton,0.20,0.25,0.55



Prix moyen des billets et répartition des passagers par port d'embarquement et par classe :



,prix_moyen,1ere_classe,2eme_classe,3eme_classe
embark_town,,,,
Cherbourg,59.95 €,50.60 %,10.12 %,39.29 %
Inconnu,80.00 €,100.00 %,0.00 %,0.00 %
Queenstown,13.28 €,2.60 %,3.90 %,93.51 %
Southampton,27.08 €,19.72 %,25.47 %,54.81 %


Le prix moyen de 80€ pour la destination inconnue est le plus élevé et s'explique surement par le fait que ce sont uniquement des voyageurs de 1ère classe. 
La plupart des voyageurs des voyageurs de Queenstown voyagent en 3ème ce qui explique que ça soit le prix moyen le plus faible.
Plus de la moitié des voyageurs de Southampton voyagent en 3ème classe ce qui peut expliquer sa 3ème position.
Enfin pour Cherbourg, la moitié des voyageurs voyagent en 1ère classe expliquant également la 2ème position.


Voyager seul augmentait-il ou diminuait-il les chances de survie ?

In [417]:
df['statut'] = np.where(
    (df['sibsp'] + df['parch']) == 0,
    'Seul',
    'Accompagné'
)

print("\n\n% de survie selon que le passager est seul ou accompagné :\n")
display(
    df.pivot_table(
        values='survived',
        index='statut',
        aggfunc='mean'
    ).mul(100).round(2)
)   



% de survie selon que le passager est seul ou accompagné :



,survived
statut,
Accompagné,50.56
Seul,30.35


On constate effectivement qu'une personne voyageant seule avait statistiquement moins de chances de survivre.